# TAGC, master notebook (v5)

## Overview
This is the one notebook to run the whole model end-to-end from this folder: config, then data, then module sanity checks, then train (`rank_mag`), then evaluate, then the long-short backtest. It uses the tuned, locked architecture and the `final_v5` feature set (adds 12-1 momentum, up-day count, and sector-rotation signals). Everything runs from `TAGC_master/` (data and venv are symlinked in).

Model: PatchTST encoder (M1) feeds the graph constructor (M2, top-5, edge-EMA), which feeds a 2-layer GAT (M3), which feeds the MC-dropout head (M4). Loss: `rank_mag` = 0.6·ListMLE + 0.4·Huber, universe-wide on the cross-sectional z-score target. lr is 3e-4.

Docs: `manuals/02_architecture_spec/ARCHITECTURE.md` (what it is), `manuals/03_user_guides/SCRIPTS.md` (every command), `manuals/01_research_log/` (the why), `experiments/` (input-feature study). Rendered doc notebooks are in `docs_notebooks/`.

## Setup

In [ ]:
import os, sys, logging
sys.path.insert(0, ".")                       # so `import model` works from here
import numpy as np, torch, pandas as pd
from model.config import Config
from model.data import build_loaders
from model.model import TAGC
from model.train import train, evaluate
logging.basicConfig(level=logging.INFO, format="%(message)s")
torch.manual_seed(0); np.random.seed(0)
print("setup ok — cwd:", os.getcwd())

## Step 1, config + data
`feature_set='final_v5'` (the v5 features) and the tuned defaults. Drop `last_n_days` / `max_epochs` for a quick smoke run, and remove them for the full thing.

In [ ]:
HORIZON = 30                                   # 5, 30 or 60

cfg = Config(feature_set="final_v5", target_horizon=HORIZON)
cfg.device = "cpu"
cfg.last_n_days = None                          # FULL 2013-2025 history (the complete v5 data). set e.g. 700 for a quick smoke
cfg.min_epochs = cfg.max_epochs = 12           # KNOW 12 is enough on CPU, bump down for a quick smoke

print("loss:", cfg.loss_kind, "| features:", len(cfg.stock_feature_columns),
      "| gat:", cfg.n_layers_gat, "| topk:", cfg.topk, "| lr:", cfg.lr,
      "| rank/mag:", cfg.rank_weight, cfg.mag_weight)

loaders = build_loaders(cfg)

print("universe:", len(loaders["tickers"]), "stocks | train days:", len(loaders["train"]),
      "| val:", len(loaders["val"]), "| test:", len(loaders["test"]))

In [9]:
loaders['tickers']

['A',
 'AAPL',
 'ABT',
 'ACGL',
 'ACN',
 'ADBE',
 'ADI',
 'ADM',
 'ADP',
 'ADSK',
 'AEE',
 'AEP',
 'AES',
 'AFL',
 'AIG',
 'AIZ',
 'AJG',
 'AKAM',
 'ALB',
 'ALGN',
 'ALL',
 'AMAT',
 'AMCR',
 'AMD',
 'AME',
 'AMGN',
 'AMP',
 'AMT',
 'AMZN',
 'AON',
 'AOS',
 'APA',
 'APD',
 'APH',
 'APO',
 'APTV',
 'ARE',
 'ATO',
 'AVB',
 'AVGO',
 'AVY',
 'AWK',
 'AXON',
 'AXP',
 'AZO',
 'BA',
 'BAC',
 'BALL',
 'BAX',
 'BBY',
 'BDX',
 'BEN',
 'BF-B',
 'BG',
 'BIIB',
 'BK',
 'BKNG',
 'BKR',
 'BLDR',
 'BLK',
 'BMY',
 'BNY',
 'BR',
 'BRK-B',
 'BRO',
 'BSX',
 'BX',
 'BXP',
 'C',
 'CAG',
 'CAH',
 'CASY',
 'CAT',
 'CB',
 'CBOE',
 'CBRE',
 'CCI',
 'CCL',
 'CDNS',
 'CF',
 'CHD',
 'CHRW',
 'CHTR',
 'CI',
 'CIEN',
 'CINF',
 'CL',
 'CLX',
 'CMCSA',
 'CME',
 'CMG',
 'CMI',
 'CMS',
 'CNC',
 'CNP',
 'COF',
 'COHR',
 'COO',
 'COP',
 'COR',
 'COST',
 'CPAY',
 'CPB',
 'CPRT',
 'CPT',
 'CRH',
 'CRL',
 'CRM',
 'CSCO',
 'CSGP',
 'CSX',
 'CTAS',
 'CTSH',
 'CVS',
 'CVX',
 'D',
 'DAL',
 'DD',
 'DE',
 'DECK',
 'DG',
 'DGX',
 'D

## Step 2, module sanity checks (M1 / M2 / M3 / M4)
Quick proof the modules are healthy (full version: `python test/m3m4_integrity_test.py`).

In [10]:
m = TAGC(cfg); m.set_universe_tickers(loaders["tickers"]); m.eval()
X_s, X_m, *_ = loaders["train"][len(loaders["train"])//2]

with torch.no_grad():
    out = m(X_s, X_m, m.init_hidden("cpu"), A_prev=m.init_adjacency("cpu"))

z = out["embeds"]
zc = z / z.norm(dim=1, keepdim=True).clamp_min(1e-8)
cos = ((zc @ zc.t()).sum() - len(zc)) / (len(zc)*(len(zc)-1))

A = out["A"]

print(f"logits shape {tuple(out['logits'].shape)} | embedding pairwise cosine {cos:.3f} (differentiated, not ~1)")
print(f"graph: {int((A>0).sum())} edges, mean out-degree {(A>0).float().sum(1).mean():.1f}")

logits shape (445,) | embedding pairwise cosine 0.096 (differentiated, not ~1)
graph: 2225 edges, mean out-degree 5.0


## Step 3, train (rank_mag) + automatic backtest
`train()` runs the loop (TBPTT + weight-EMA + best-on-val-Rank-IC), writes universe-wide `predictions_test.csv`, and runs the long-short backtest at the end. This is the slow cell.

In [11]:
out_dir = f"runs_final/master_{HORIZON}d"
res = train(cfg, out_dir=out_dir, resume=False)
print("done. test Rank-IC:", round(float(res["test_mc"].rank_ic_mean), 4))

target check: return_30d  regression_scale=1.0 → scaled std=1.000 (raw≈1.0000)
target=AAPL (idx 1)  n_stocks=445  n_macro=10  train=2067  val=321  test=642
v3 regression: target=return_30d (FORWARD 30-day cumulative)  scale=1.0
────────────────────────────────────────────────────────────────
Ljung-Box on training-period AAPL close_ret_raw (n=2247)
Expect: p>>0.05 for raw (no mean predictability);
        p~0    for abs / sq (volatility clusters → ARCH).
transform   lag     lb_stat     p_value
   raw     1      11.775   0.0006003
   raw     5      12.814     0.02519
   raw    10      82.403   1.694e-13
   raw    20     113.758   4.035e-15
   abs     1     100.444   1.218e-23
   abs     5     399.530   3.747e-84
   abs    10     734.891  2.022e-151
   abs    20    1016.639  1.104e-202
    sq     1     149.384   2.363e-34
    sq     5     440.559    5.34e-93
    sq    10     782.323  1.301e-161
    sq    20     960.335  1.114e-190
──────────────────────────────────────────────────────────

done. test Rank-IC: 0.0118


## Step 4, results: backtest + figures

In [12]:
bt = pd.read_csv(f"{out_dir}/backtest_summary.csv").T
print(bt)
# figures: python scripts/visualize.py runs_final/master_30d
# interactive graph: python scripts/interactive_graph.py runs_final/master_30d

                                0
horizon                 30.000000
decile                   0.100000
n_days                 642.000000
n_periods_nonoverlap    22.000000
avg_K                  439.809969
rank_ic_mean             0.011600
rank_ic_std              0.104020
rank_icir                0.111500
dir_acc                  0.499000
ls_ret_mean_period      -0.005670
ls_ret_std_period        0.033460
ls_sharpe_overlap       -0.194000
ls_sharpe_nonoverlap    -0.491000
ls_hit_rate              0.545500
ls_hit_rate_overlap      0.486000
cum_return_compounded   -0.128100
cum_return_additive     -0.124700


In [13]:
!python scripts/visualize.py runs_final/master_30d

target = AAPL
wrote 6 figures to runs_final/master_30d/figures:
  - 1_training_curves.png
  - 2_ranking_overview.png
  - 4_graph_summary.png
  - 5_top_neighbours.png
  - 7_graph_network.png
  - 8_full_graph.png


In [14]:
!python scripts/interactive_graph.py runs_final/master_30d

target  = AAPL
frames  = 642 (2023-03-15 → 2025-10-03)
sectors = 11 present
showing = average over 642 test days   ε = 0.020
wrote   runs_final/master_30d/figures/interactive_graph.html  (1.7 MB)
open    file:///Users/oscarw/Documents/github/uni-master-tagc-2026/runs_final/master_30d/figures/interactive_graph.html


## Step 4b, check predictions vs reality
`predictions_test.csv` is the raw per-stock output (one row per date×stock): the model's
`pred_score`/`pred_rank`/`pred_direction` next to the REAL `true_fwd_return`/`true_rank`, plus a
`direction_correct` flag. This cell loads it and compares predicted vs real.

In [17]:
import pandas as pd, numpy as np

pred = pd.read_csv(f"{out_dir}/predictions_test.csv", parse_dates=["date"])

print(f"{len(pred):,} predictions | {pred.date.nunique()} test days | {pred.ticker.nunique()} stocks")
print("\nsample rows (predicted vs real):")

cols = ["date","ticker","pred_score","pred_rank","pred_direction","true_fwd_return","true_rank","direction_correct"]

print(pred.sample(8, random_state=0)[cols].to_string(index=False))

# headline predicted-vs-real metrics
ic = pred.groupby("date").apply(lambda d: d["pred_rank"].corr(d["true_rank"])).mean()

print("\nmean daily Rank-IC (pred_rank vs true_rank):", round(float(ic),4))
print("direction accuracy:", round(float(pred.direction_correct.mean()),3))

dec = pred.groupby("date").apply(lambda d: pd.Series({
    "top_decile_real":    d.loc[d.pred_rank>=0.9, "true_fwd_return"].mean(),
    "bottom_decile_real": d.loc[d.pred_rank<=0.1, "true_fwd_return"].mean()})).mean()

print("avg REAL fwd-return of predicted TOP decile:", round(float(dec.top_decile_real),4),
      "| BOTTOM decile:", round(float(dec.bottom_decile_real),4),
      "(top should beat bottom if predictions track reality)")

282,358 predictions | 642 test days | 442 stocks

sample rows (predicted vs real):
      date ticker  pred_score  pred_rank  pred_direction  true_fwd_return  true_rank  direction_correct
2024-01-25     PM    0.012345   0.961098               1         0.039185   0.430206                  0
2025-08-05    CMG   -0.035714   0.226757               0        -0.067463   0.074830                  1
2025-09-26    DVN    0.007336   0.814059               1        -0.045045   0.360544                  0
2024-11-04    BMY   -0.085376   0.004545               0         0.015907   0.720455                  0
2023-05-03    AMP    0.002851   0.559633               1         0.157944   0.876147                  1
2025-06-23    REG    0.010283   0.970522               1         0.021490   0.507937                  1
2023-03-29   PANW   -0.084320   0.006881               0         0.029602   0.605505                  0
2023-09-21    DUK    0.008751   0.733945               1        -0.032680   0.295872 

## Where to go next
- Different inputs: `feature_set='final'` (33 baseline) vs `'final_v5'` (v5). See `experiments/2026-06-14_input_feature_screen/`.
- Ablations: `cfg.model_variant` in {`tagc`, `no_gru`, `no_gat`, `static_graph`, `no_graph`, `random_graph`}.
- Re-tune: run `python scripts/hpo_rank_mag.py`, which writes to `hyperparam/`.
- All commands: `manuals/03_user_guides/SCRIPTS.md`.

Honest expectation: a small, regime-dependent edge. On the recent test regime Rank-IC sits near 0 (the documented data ceiling), and the model reaches a higher ceiling where signal exists (pre-2020). Report a regime-aware edge plus the backtest, not a single number.

---
# Experiments (scratchpad)
Appended tests, run after the cells above (they reuse `cfg`, `loaders`, `out_dir`, `train`, `build_loaders`). Quick checks, not full retrains unless noted.

## E1, daily learned graph (interactive, per-day, no averaging)
Animates the learned adjacency day by day (fixed node layout so you watch edges change, not nodes jump). Use the slider to pick a day, then use the plotly toolbar camera icon to save a PNG for the thesis.

In [ ]:
import os, glob, numpy as np, torch, networkx as nx, plotly.graph_objects as go
from model.model import TAGC
from model.train import evaluate
from model.graph_logger import GraphLogger

RUN = out_dir                       # runs_final/master_30d, from the train cell
tickers = loaders["tickers"]; N = len(tickers)
gdir = f"{RUN}/graphs"
snaps = sorted(glob.glob(f"{gdir}/test_*.npz"))
if len(snaps) < 5:                  # regenerate per-day snapshots from best.pt (few min)
    mdl = TAGC(cfg); mdl.set_universe_tickers(tickers)
    ck = torch.load(f"{RUN}/best.pt", map_location="cpu", weights_only=False)
    mdl.load_state_dict(ck["model_state"], strict=False); mdl.eval()
    evaluate(mdl, loaders["test"], cfg, graph_logger=GraphLogger(gdir, tickers=tickers, every=5),
             split_name="test")
    snaps = sorted(glob.glob(f"{gdir}/test_*.npz"))
snaps = snaps[-40:]                 # most recent ~40 saved days
days = [np.load(s, allow_pickle=True) for s in snaps]
print(f"{len(days)} daily graphs: {str(days[0]['date'])} -> {str(days[-1]['date'])}")

# fixed layout from the mean adjacency (positions only; edges are still per-day)
Amean = np.mean([d["adj"] for d in days], axis=0)
G = nx.from_numpy_array((Amean > np.percentile(Amean, 99)).astype(float))
pos = nx.spring_layout(G, seed=1, k=1.5/np.sqrt(N))
xn = np.array([pos[i][0] for i in range(N)]); yn = np.array([pos[i][1] for i in range(N)])
tgt = cfg.target_idx

def edges(A):
    thr = np.percentile(A[A > 0], 92) if (A > 0).any() else 1.0   # TODO maybe expose this percentile as a knob; that day's strongest edges
    ex, ey = [], []
    for i, j in zip(*np.where(A > thr)):
        ex += [xn[i], xn[j], None]; ey += [yn[i], yn[j], None]
    return go.Scatter(x=ex, y=ey, mode="lines", line=dict(width=0.4, color="#bbb"), hoverinfo="none")

nodes = go.Scatter(x=xn, y=yn, mode="markers", text=tickers, hoverinfo="text",
                   marker=dict(size=[10 if i==tgt else 5 for i in range(N)],
                               color=["#d62728" if i==tgt else "#1f77b4" for i in range(N)]))
frames = [go.Frame(data=[edges(d["adj"]), nodes], name=str(d["date"])) for d in days]
fig = go.Figure(data=[edges(days[-1]["adj"]), nodes], frames=frames)
fig.update_layout(title="Daily learned graph (per-day, not averaged) — slide to a date, camera icon to screenshot",
    width=820, height=720, showlegend=False, xaxis=dict(visible=False), yaxis=dict(visible=False),
    updatemenus=[dict(type="buttons", x=0, y=0, buttons=[dict(label="play", method="animate", args=[None])])],
    sliders=[dict(steps=[dict(method="animate", args=[[f.name]], label=f.name) for f in frames])])
fig.show()


## E2, pre-COVID dataset (2013-2020)
Builds the model on the pre-COVID window only (where the cross-sectional signal actually exists). Quick check confirms the split, and the `train(...)` call runs the real thing (about 1 h on CPU, so bump epochs down for a smoke).

In [ ]:
from model.config import Config
cfg_pre = Config(feature_set="final_v5", target_horizon=30)
cfg_pre.device = "cpu"
cfg_pre.start_date = "2013-01-01"; cfg_pre.end_date = "2020-01-01"   # pre-COVID
cfg_pre.min_epochs = cfg_pre.max_epochs = 12
L_pre = build_loaders(cfg_pre)
for s in ["train", "val", "test"]:
    d = L_pre[s]; print(f"{s:5s}: {d[0][6]} -> {d[len(d)-1][6]}  ({len(d)} days)")   # KNOW index 6 is the date field in the tuple
print("\npre-COVID dataset ready. To train on it (real run):")
print("   res_pre = train(cfg_pre, out_dir='runs_final/precovid_30d', resume=False)")
# quick smoke instead of the full hour:  cfg_pre.last_n_days=700; cfg_pre.min_epochs=cfg_pre.max_epochs=3


In [23]:
res_pre = train(cfg_pre, out_dir='runs_final/precovid_30d', resume=False)

target check: return_30d  regression_scale=1.0 → scaled std=1.000 (raw≈1.0000)
target=AAPL (idx 1)  n_stocks=445  n_macro=10  train=1053  val=176  test=353
v3 regression: target=return_30d (FORWARD 30-day cumulative)  scale=1.0
────────────────────────────────────────────────────────────────
Ljung-Box on training-period AAPL close_ret_raw (n=2247)
Expect: p>>0.05 for raw (no mean predictability);
        p~0    for abs / sq (volatility clusters → ARCH).
transform   lag     lb_stat     p_value
   raw     1      11.775   0.0006003
   raw     5      12.814     0.02519
   raw    10      82.403   1.694e-13
   raw    20     113.758   4.035e-15
   abs     1     100.444   1.218e-23
   abs     5     399.530   3.747e-84
   abs    10     734.891  2.022e-151
   abs    20    1016.639  1.104e-202
    sq     1     149.384   2.363e-34
    sq     5     440.559    5.34e-93
    sq    10     782.323  1.301e-161
    sq    20     960.335  1.114e-190
──────────────────────────────────────────────────────────

## E3, volatility target: is risk more predictable than return?
Ljung-Box said volatility clusters (predictable) while direction doesn't. Quick cross-sectional check: predict forward 30-day realized volatility from current realized vol (vol persistence) vs predicting forward return. If vol-IC is much bigger than return-IC, a vol/risk target is the forecastable one.

In [24]:
import pandas as pd, numpy as np
dfv = pd.read_parquet("data/feature_experiments/stocks_30d_v5.parquet",
                      columns=["close_ret", "return_30d"]).reset_index().sort_values(["ticker","date"])
g = dfv.groupby("ticker", group_keys=False)
dfv["fwd_vol_30"] = g["close_ret"].apply(lambda s: s.rolling(30).std().shift(-30))   # the vol TARGET
dfv["cur_vol_21"] = g["close_ret"].apply(lambda s: s.rolling(21).std())              # predictor

def xs_rankic(frame, feat, tgt):
    s = frame[["date", feat, tgt]].replace([np.inf,-np.inf], np.nan).dropna()
    rf = s.groupby("date")[feat].rank(); rt = s.groupby("date")[tgt].rank()
    t = pd.DataFrame({"d": s.date.values, "rf": rf.values, "rt": rt.values}); gg = t.groupby("d")
    a = t.rf - gg.rf.transform("mean"); b = t.rt - gg.rt.transform("mean")
    return float(((a*b).groupby(t.d).sum() /
                  np.sqrt((a**2).groupby(t.d).sum()*(b**2).groupby(t.d).sum())).dropna().mean())

print("cross-sectional Rank-IC, current 21d vol  ->  forward 30d VOLATILITY:",
      round(xs_rankic(dfv, "cur_vol_21", "fwd_vol_30"), 4), "  <- the predictable target")
print("cross-sectional Rank-IC, current 21d vol  ->  forward 30d RETURN    :",
      round(xs_rankic(dfv, "cur_vol_21", "return_30d"), 4), "  <- return ~ noise")
print("\n=> if vol-IC is large and positive, retarget the model to forward vol "
      "(set the target column to fwd_vol_30 and retrain) — a strong secondary thesis result.")


cross-sectional Rank-IC, current 21d vol  ->  forward 30d VOLATILITY: 0.1368   <- the predictable target
cross-sectional Rank-IC, current 21d vol  ->  forward 30d RETURN    : -0.0043   <- return ~ noise

=> if vol-IC is large and positive, retarget the model to forward vol (set the target column to fwd_vol_30 and retrain) — a strong secondary thesis result.


## E4, validation across regimes (per-year Rank-IC, not just the recent tail)
Instead of one chronological test on 2023-2025, score the signal in every year. That's a walk-forward-style view that shows where the signal lives (strong pre-2020, decays after).

In [21]:
import pandas as pd, numpy as np
dfy = pd.read_parquet("data/feature_experiments/stocks_30d_v5.parquet",
                      columns=["sector_mom21", "return_30d"]).reset_index()
dfy["year"] = pd.to_datetime(dfy.date).dt.year
def ic_year(frame, feat):
    res = {}
    for y, sub in frame.groupby("year"):
        s = sub[["date", feat, "return_30d"]].replace([np.inf,-np.inf], np.nan).dropna()
        if len(s) < 200: continue
        rf = s.groupby("date")[feat].rank(); rt = s.groupby("date")["return_30d"].rank()
        t = pd.DataFrame({"d": s.date.values, "rf": rf.values, "rt": rt.values}); gg = t.groupby("d")
        a = t.rf - gg.rf.transform("mean"); b = t.rt - gg.rt.transform("mean")
        res[y] = float(((a*b).groupby(t.d).sum() /
                        np.sqrt((a**2).groupby(t.d).sum()*(b**2).groupby(t.d).sum())).dropna().mean())
    return pd.Series(res)
icy = ic_year(dfy, "sector_mom21")
print("cross-sectional Rank-IC of sector_mom21 by year:\n", icy.round(4).to_string())
import matplotlib.pyplot as plt
ax = icy.plot(kind="bar", figsize=(9,3), color=["#2ca02c" if v>0 else "#d62728" for v in icy])
ax.axhline(0, color="k", lw=0.8); ax.set_ylabel("Rank-IC")
ax.set_title("Rank-IC by year (strongest single feature) — signal pre-2020, decays after")
plt.tight_layout(); plt.show()


cross-sectional Rank-IC of sector_mom21 by year:
 2013   -0.0731
2014   -0.0079
2015   -0.0542
2016    0.0096
2017   -0.0850
2018   -0.0354
2019   -0.0673
2020    0.0396
2021   -0.0754
2022   -0.0530
2023   -0.0004
2024   -0.0025
2025   -0.0299


/var/folders/ms/5j9g7g1n0rsclq3dbv4kz_h80000gn/T/ipykernel_42219/3906675006.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
